In [12]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv

from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w2_token_embedding_vector_rag/llm_260317_Tokens_Embeddings.ipynb)

In [13]:
#!pip install langchain-openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# Embeddings:
# 이미지, 영상, 텍스트, 음성 -> vector space
# 고차원 데이터의 저차원 압축 표현
# Image (3, 1024, 1024) -> (32, 32) 저차원의 압축 매트릭스로 저장?표현?
# Text (100, 768, 50000) -> ~~~

In [30]:
llm = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)

In [15]:
test_emb = embeddings_model.embed_query("Hello world")
len(test_emb)
#1536

1536

In [16]:
test_emb[:10]

[-0.0021610260009765625,
 -0.049072265625,
 0.0209808349609375,
 0.0313720703125,
 -0.045318603515625,
 -0.0264129638671875,
 -0.028961181640625,
 0.060302734375,
 -0.0257110595703125,
 -0.0148162841796875]

In [18]:
# !pip install tiktoken
import tiktoken

In [19]:
# embedding space, representation space 모델의 세계관
enc = tiktoken.encoding_for_model('gpt-4o-mini')

In [24]:
text = '안녕하세요. 오늘 LLM에 대해 배워볼게요!'
tokens = enc.encode(text)
tokens

[14307,
 171731,
 13,
 106820,
 451,
 19641,
 3107,
 67946,
 33628,
 33771,
 70785,
 7996,
 7952,
 0]

In [25]:
enc.decode(tokens)

'안녕하세요. 오늘 LLM에 대해 배워볼게요!'

In [29]:
# Chunking
# model이 가진 텍스트 id
# -> API 비용이 1M당 input이 $0.05 output $0.2
# 즉, 이때 지칭하는 token이 아래와 같음 (12개의 토큰..)
for i, token_id in enumerate(tokens):
  token_text = enc.decode([token_id])
  print(f' token {i+1} : ID : {token_id} -> {token_text}')

 token 1 : ID : 14307 -> 안
 token 2 : ID : 171731 -> 녕하세요
 token 3 : ID : 13 -> .
 token 4 : ID : 106820 ->  오늘
 token 5 : ID : 451 ->  L
 token 6 : ID : 19641 -> LM
 token 7 : ID : 3107 -> 에
 token 8 : ID : 67946 ->  대해
 token 9 : ID : 33628 ->  배
 token 10 : ID : 33771 -> 워
 token 11 : ID : 70785 -> 볼
 token 12 : ID : 7996 -> 게
 token 13 : ID : 7952 -> 요
 token 14 : ID : 0 -> !


In [ ]:
# 왜 1글자, 4글자로 규칙성 없이 쪼개지는 것처럼 보일까?
# 현대 LLM의 tokenizing 방식의 차이..
# I like go to school -> I / like / go / to / school
# He likes go to school -> ?
# ...
# voca {'I': 1000, 'like': 211, ...} likes는?? # <unk> <unk> go to school 정확도가 떨어짐..
# 그래서 BPE Byte Pair Encoding 방식으로 나눔
# 안 녕 하 세 요. -> 안녕 하 세 요 -> 안녕 하세요.. 자주 나오는 방식으로 구성
# He likes go to school -> He, like, + s, go, to, school (BPS 방식)

In [32]:
# Practice
# 요즘은 context window가 많이 커졌지만.. 기본적인 LLM이 긴 문서를 처리하는 방식은 아래와 같아.
long_text = """인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다."""

'인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.\n최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.\nLangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.\nRAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.'

In [ ]:
# chunk
# Splitter 긴 텍스트, 문서를 chunk 단위로 나눠줌
# 우리 모델에 맞는 context window에 맞춰서 잘라줌
# CharacterTextSplitter(character)
# RecursiveCaracterSplitter(token)
# from tiktoken_encoder()

In [34]:
!pip install langchain_text_splitters

In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [47]:
# chunk_size: 나누는 문맥 사이즈
# chunk_overlap: 앞 뒤 문맥을 파악할 수 있도록 오버랩 (chunk_size보다 작게)
# seperators = []
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 20, separators = ['\n\n', '\n', '. ', ' ', ''])
chunks = splitter.split_text(long_text)
len(chunks)

7

In [50]:
# RecursiveCharacterTextSplitter 특징
# chunk_size가 100이어도 문장을 보기 때문에 문장 단위로 자른다.
splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap = 20, separators = ['\n\n', '\n', '. ', ' ', ''])
chunks = splitter.split_text(long_text)
for i, chunk in enumerate(chunks):
  print(f'[chunk {i+1}] : {len(chunk)} characters')
  print(chunk)

[chunk 1] : 46 characters
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을
[chunk 2] : 36 characters
학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 3] : 46 characters
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다
[chunk 4] : 50 characters
. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인
[chunk 5] : 31 characters
자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 6] : 46 characters
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다
[chunk 7] : 40 characters
. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며,
[chunk 8] : 42 characters
벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented
[chunk 9] : 29 characters
Generation) 시스템 구축에 특히 유용합니다.
[chunk 10] : 40 characters
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
[chunk 11] : 49 characters
. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여
[chunk 12] : 31 characters
후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [52]:
def count_token(text, model = 'gpt-4o-mini'):
  enc = tiktoken.encoding_for_model(model)
  return len(enc.encode(text))

In [56]:
# RecursiveCharacterTextSplitter 토큰으로 자르기
splitter_tiktoken = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name = 'gpt-4o-mini', chunk_size = 50, chunk_overlap=10
)
chunks = splitter_tiktoken.split_text(long_text)
for i, chunk in enumerate(chunks):
  tokens = count_token(chunk)
  print(f'[chunk {i+1}] : {len(chunk)} characters, 실제 토큰: {tokens} tokens')
  print(chunk)

[chunk 1] : 65 characters, 실제 토큰: 40 tokens
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 2] : 100 characters, 실제 토큰: 47 tokens
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를
[chunk 3] : 27 characters, 실제 토큰: 15 tokens
처리 분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 4] : 80 characters, 실제 토큰: 47 tokens
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을
[chunk 5] : 72 characters, 실제 토큰: 27 tokens
벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
[chunk 6] : 84 characters, 실제 토큰: 49 tokens
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를
[chunk 7] : 31 characters, 실제 토큰: 17 tokens
후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [57]:
# 한국어 test_text를 chunking 해보세요. 다양한 chunk size, overlap size를 이용
test_text = """서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다."""

In [66]:
# 캐릭터(자수) 단위로 나누기
splitter_char = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap = 10, separators = ['\n\n', '\n', ', ',  ' ', ''])
chunks_char = splitter_char.split_text(test_text)
for i, chunk in enumerate(chunks_char):
  print(f'[chunk {i+1}] : {len(chunk)} characters')
  print(chunk)
print(' ========== ')
# token 단위로 나누기
splitter_token = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name = 'gpt-4o-mini', chunk_size = 100, chunk_overlap=10
)
chunks_token = splitter_token.split_text(test_text)
for i, chunk in enumerate(chunks_token):
  tokens = count_token(chunk)
  print(f'[chunk {i+1}] : {len(chunk)} characters, 실제 토큰: {tokens} tokens')
  print(chunk)

[chunk 1] : 46 characters
서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며
[chunk 2] : 36 characters
, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁
[chunk 3] : 22 characters
, 창덕궁 등 조선시대 궁궐과 N서울타워
[chunk 4] : 48 characters
, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한
[chunk 5] : 46 characters
세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의
[chunk 6] : 31 characters
노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다.
[chunk 1] : 152 characters, 실제 토큰: 97 tokens
서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한
[chunk 2] : 76 characters, 실제 토큰: 45 tokens
면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다.


In [67]:
# 실습 선생님 답안
def split_and_report(text, chunk_size=50, chunk_overlap=10):
  splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
      model_name = 'gpt-4o-mini', chunk_size=chunk_size, chunk_overlap=chunk_overlap
  )
  chunk = splitter.split_text(text)
  token_counts = [count_token(c) for c in chunks]

  print(f'chunking report (chunk_size = {chunk_size}, chunk_overlap = {chunk_overlap})')
  for i, (chunk, tc) in enumerate(zip(chunks, token_counts)):
    first_line = chunk.split('\n')[0][:40]
    print(f' [{i+1}] {tc} tokens, {len(chunk)} characters | {first_line}')

  print(f'summary : {len(chunks)} chunks')
  print(f'average : {np.mean(token_counts)}')
  print(f'max : {max(token_counts)}')

In [69]:
split_and_report(test_text)

chunking report (chunk_size = 50, chunk_overlap = 10)
 [1] 40 tokens, 65 characters | 인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력,
 [2] 47 tokens, 100 characters | 최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪
 [3] 15 tokens, 27 characters | 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
 [4] 47 tokens, 80 characters | LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레
 [5] 27 tokens, 72 characters | 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augment
 [6] 49 tokens, 84 characters | RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
 [7] 17 tokens, 31 characters | 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.
summary : 7 chunks
average : 34.57142857142857
max : 49


In [71]:
# Embedding 모델 (aka 세계관)
# 우리가 학습시킨 모델이 구성한 1536차원의 세계..
# 그 세계관 안에서 단어들의 관계들이 이미 구성되어 있다.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [31]:
embedding_model = OpenAIEmbeddings(model = 'text-embedding-3-small', api_key = api_key)
# embedding_model('hello')

In [73]:
query = '인공지능이 세상을 바꾸고 있습니다.'
query_vector = embeddings_model.embed_query(query)
len(query_vector)

1536

In [ ]:
# 15000: 주어와 목적어 사이의 관계.. 주어랑 다음에 올지 모르는 단위와 동사의 관계.. 등
# 1500: 주어, 목적어, 동사, 그들의 관계..

In [91]:
text = [
    "AI가 세상을 바꾸고 있습니다.",
    'AI는 인공지능입니다.',
    '머신러닝으로 질병을 예측할 수 있다.',
    '치킨 먹고 싶어요'
]

doc_vectors = embeddings_model.embed_documents(text)
len(doc_vectors), len(doc_vectors[0])

for vec in doc_vectors:
  print(len(vec))

1536
1536
1536
1536


In [80]:
doc_vectors[0]

[0.02435302734375,
 0.0301513671875,
 -0.019195556640625,
 0.036834716796875,
 0.02410888671875,
 -0.01439666748046875,
 -0.0177459716796875,
 0.046356201171875,
 -0.063232421875,
 -0.0228729248046875,
 -0.0537109375,
 -0.01473236083984375,
 0.033660888671875,
 -0.054351806640625,
 -0.03985595703125,
 -0.01546478271484375,
 -0.08319091796875,
 0.0207977294921875,
 0.0692138671875,
 -0.018798828125,
 -0.00099945068359375,
 0.006587982177734375,
 0.012420654296875,
 -0.010284423828125,
 0.03558349609375,
 -0.01114654541015625,
 0.04150390625,
 0.062042236328125,
 0.01629638671875,
 -0.00977325439453125,
 0.044586181640625,
 -0.03228759765625,
 -0.0264892578125,
 -0.0119171142578125,
 0.00022363662719726562,
 0.0228118896484375,
 0.0186767578125,
 0.003566741943359375,
 -0.0193023681640625,
 -0.0154571533203125,
 0.01251220703125,
 -0.007373809814453125,
 0.053863525390625,
 0.0325927734375,
 0.0128936767578125,
 0.010711669921875,
 -0.064453125,
 -0.0134429931640625,
 0.06719970703125,
 

In [22]:
# Embedding을 이용해서 유사도를 확인할 때
# cosign similarity 벡터간의 유사 정도를 파악함
# 사잇 각이 가까워지면 코사인 값이 커지고, 멀어지면 코사인 값이 작아진다. -> 유사도를 파악할 수 있음
# !pip install scikit-learn
from sklearn.metrics.pairwise import cosine_similarity

In [83]:
cosine_similarity(doc_vectors)
# 대각선을 중심으로 대칭이다..!
# 행/열이 1,2,3,4 벡터의 관계를 알 수 있다. -1(유사도 낮음) ~ 1(유사도 높음)

array([[1.        , 0.56734978, 0.16457626, 0.33995876],
       [0.56734978, 1.        , 0.1787882 , 0.14887418],
       [0.16457626, 0.1787882 , 1.        , 0.13672291],
       [0.33995876, 0.14887418, 0.13672291, 1.        ]])

In [94]:
# cosine 유사도로 query에 대해 답변 유사도를 확인할 수 있다.
all_vectors = [query_vector] + doc_vectors
all_texts = [query] + text

cosine_similarity(all_vectors)
# 인공지능이 세상을 바꾸고 있습니다.-> AI가 세상을 바꾸고 있습니다. 가장 유사함.

array([[1.        , 0.77406712, 0.68264085, 0.21886244, 0.13006756],
       [0.77406712, 1.        , 0.56733416, 0.1645455 , 0.19547004],
       [0.68264085, 0.56733416, 1.        , 0.17880587, 0.17766485],
       [0.21886244, 0.1645455 , 0.17880587, 1.        , 0.14956247],
       [0.13006756, 0.19547004, 0.17766485, 0.14956247, 1.        ]])

In [92]:
similarity_matrix = cosine_similarity(all_vectors)
labels = ['query'] + [f'doc{i+1}' for i in range(len(text))]

df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
print(df)

          query      doc1      doc2      doc3      doc4
query  1.000000  0.774133  0.682669  0.218879  0.280374
doc1   0.774133  1.000000  0.567350  0.164576  0.339959
doc2   0.682669  0.567350  1.000000  0.178788  0.148874
doc3   0.218879  0.164576  0.178788  1.000000  0.136723
doc4   0.280374  0.339959  0.148874  0.136723  1.000000


In [23]:
# 질문이 어떤 카테고리와 가장 유사한지, 카테고리 분류하는 함수를 작성해 보세요.
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}

# 카테고리 텍스트 벡터화
categories_vectors = {}
for label, examples in categories.items():
    categories_vectors[label] = embeddings_model.embed_documents(examples)

def classify(text, categories_vectors, embeddings_model):
  # 가장 유사도 높은 카테고리 구분하기
  query_vector = embeddings_model.embed_query(text)

  best_category = None
  max_similarity = -1

  for label, example_vectors in categories_vectors.items():
    # Calculate similarity between query and all example vectors in the current category
    similarities = cosine_similarity([query_vector], example_vectors)[0]

    # Calculate the average similarity for the category
    avg_similarity = np.mean(similarities)

    if avg_similarity > max_similarity:
      max_similarity = avg_similarity
      best_category = label

  return best_category

query = '프로야구 시즌은?'
print(classify(query, categories_vectors, embeddings_model))

스포츠


In [25]:
# 선생님 답안
def classify(text, categories):
  cat_vectors = {}
  for cat, examples in categories.items():
    embs = embeddings_model.embed_documents(examples)
    cat_vectors[cat] = np.mean(embs, axis = 0)

  text_emb = embeddings_model.embed_query(text)

  scores = {}
  for cat, cat_vec in cat_vectors.items():
    sim = cosine_similarity([text_emb], [cat_vec])[0][0]
    scores[cat] = sim

  best_cat = max(scores, key=scores.get)

  print(f"입력 : {text}")
  print(f"예측 : {best_cat}")
  for c, s in scores.items():
      marker = " <<<" if c == best_cat else ""
      print(f" {c}: {s}{marker}")

  return best_cat, scores

test_sentense = ['새로운 GPU가 출시되어 인터넷이 빨라집니다.', '올해 올림픽에서 금메달 따면 좋겠어요.']

for sent in test_sentense:
  classify(sent, categories)
  print('---')

입력 : 새로운 GPU가 출시되어 인터넷이 빨라집니다.
예측 : 기술
 기술: 0.34241015892681304 <<<
 스포츠: 0.22686988225727853
 음식: 0.15264139336917487
---
입력 : 올해 올림픽에서 금메달 따면 좋겠어요.
예측 : 스포츠
 기술: 0.09416179238410832
 스포츠: 0.5371110204482162 <<<
 음식: 0.19172538417153057
---


In [26]:
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}

test_senteces = ["새로운 GPU가 출시되어 AI 학습속도가 빨라졌습니다", "올해 올림픽에서 한국이 좋은 성적을 거뒀다", "이 식당 불고기가 정말 맛있다"]

for sent in test_senteces:
    print()
    classify(sent, categories)


입력 : 새로운 GPU가 출시되어 AI 학습속도가 빨라졌습니다
예측 : 기술
 기술: 0.38362292492778444 <<<
 스포츠: 0.22169883463842033
 음식: 0.11053494030252108

입력 : 올해 올림픽에서 한국이 좋은 성적을 거뒀다
예측 : 스포츠
 기술: 0.15820029749117687
 스포츠: 0.5189469292546451 <<<
 음식: 0.3005985258795071

입력 : 이 식당 불고기가 정말 맛있다
예측 : 음식
 기술: 0.13398171521542115
 스포츠: 0.17112716538351494
 음식: 0.46908468001114206 <<<


In [27]:
# Embedding Cache "CacheBackedEmbeddings"
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_core.stores import InMemoryByteStore

In [32]:
store = InMemoryByteStore()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
                    embedding_model, store, namespace='embedding-cache')

/usr/local/lib/python3.12/dist-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()
